# Basic API usage example

## Overview

The goal of this tutorial is to wire up a simple model that is comprised of a spatially embedded brain area, specify a simple learning objective, and optimize model parameters.

Before we begin, here are a few notes pertaining to the nomenclature we have adopted.

- A neuron ***class*** is defined by its synaptic affiliation. In `torch-biopl` you can configure types to be `Excitatory`/`Inhibitory` (where synapses have a postive/negative sign), or `Hybrid` which defaults to standard machine learning-style synapses that are unconstrained.
- Within each neuron class, you can instantiate neuron ***types***. We employ the definition of neuron types with an eye to be able to specify inter-type local connectivity rules. 
- Within each neuron type, are ***subtypes***. Neurons within a subtype can (but don't have to) share properties like time constants, nonlinearities, and biases.
- Each neural area can be configured independently by specifying its classes, types, subtypes, and inter-type connectivity rules.
- Areas can be stitched together to form larger networks.
- Learnable parameters in `torch-biopl` are usually the synaptic weights, neural time constants, and biases.

In [7]:
import numpy as np
import torch
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from tqdm import tqdm
from bioplnn.datasets import Mazes

maze_data_path = "/om2/user/jackking/torch-bioplnn-dev/data/mazes"

from bioplnn.models import SpatiallyEmbeddedClassifier

In [2]:
# Torch setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

Let us wire up a simple one-area network with two neural classes. Let one of them be an `Excitatory` cell class and one be `Inhibitory` each with `16` subtypes. Now, we have our neural populations. All that's left to do is to specify the inter-type connectivity. 

In `torch-biopl` we adopt the following convention:

- Inter-celltype connectivity (within a given area) is specified through an adjacency matrix.
- In addition to the neuron types within an area, we also have to account for projections into and out of the area. Keeping this in mind, we use a schema where rows in the adjacency matrix represent the ***pre-synaptic*** neuron type and columns represent the ***post-synaptic*** neuron type. 
- The **first row** always denotes projections into the area, and the **last column** always denotes feedforward projections out of the area.

For example, if our neuron_type_1 is E and neuron_type_2 is I, then `inter_neuron_type_connectivity` = $\big(\begin{smallmatrix} 1 & 1 & 0 \cr 1 & 1 & 1 \cr 1 & 1 & 0 \end{smallmatrix}\big)$ represents a standard recurrent inhibitory circuit motif (ala [Wong et al. (2006)](https://pubmed.ncbi.nlm.nih.gov/16436619/)), where both the E and I populations receive input, and only the E population projects downstream.

Since we plan to train on grayscale images in this example, `in_channels` = 1

In [11]:
# Model setup
model = SpatiallyEmbeddedClassifier(
    rnn_kwargs={
        "num_areas": 1,
        "area_kwargs": [
            {
                "num_neuron_types": 2,
                "num_neuron_subtypes": np.array([16, 16]),
                "neuron_type_class": np.array(["excitatory", "inhibitory"]),
                "inter_neuron_type_connectivity": np.array(
                    [[1, 1, 0], [1, 1, 1], [1, 1, 0]]
                ),
                "in_size": [48, 48],
                "in_channels": 4,
                "out_channels": 32,
            },
        ],
    },
    num_classes=10,
    fc_dim=256,
    dropout=0.5,
).to(device)

In [12]:
# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define the loss function
criterion = nn.CrossEntropyLoss()

In [13]:
# Dataloader setup
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train_data = MNIST(root="data", train=True, transform=transform, download=True)
train_loader = DataLoader(
    train_data, batch_size=256, num_workers=8, shuffle=True
)

/om2/user/jackking/anaconda/envs/bioplnn/lib/python3.12/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [14]:
class MazeDataset(torch.utils.data.Dataset):
    def __init__(self, maze_data_path, transform=None):
        self.mazes = Mazes(maze_data_path)
        self.transform = transform
        
    def __len__(self):
        return len(self.mazes)
    
    def __getitem__(self, idx):
        maze, label = self.mazes[idx]
        if self.transform:
            maze = self.transform(maze)
        return maze, label

In [15]:
maze_dataset = MazeDataset(maze_data_path)
num_samples = 20000
if num_samples is None:
    num_samples = len(maze_dataset)
indices = torch.randperm(len(maze_dataset))[:num_samples].tolist()
maze_dataset = torch.utils.data.Subset(maze_dataset, indices)

# Split dataset into train and test sets
train_size = int(0.8 * num_samples)
test_size = num_samples - train_size
train_dataset, test_dataset = torch.utils.data.random_split(maze_dataset, [train_size, test_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [16]:
# Define the training loop
model.train()
n_epochs = 10
log_frequency = 100

running_loss, running_correct, running_total = 0, 0, 0
for epoch in range(n_epochs):
    for i, (x, labels) in enumerate(tqdm(train_loader)):
        x = x.to(device)
        labels = labels.to(device)
        torch._inductor.cudagraph_mark_step_begin()
        logits = model(x, num_steps=5)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate running accuracy and loss
        _, predicted = torch.max(logits, 1)
        running_total += labels.size(0)
        running_correct += (predicted == labels).sum().item()
        running_loss += loss.item()

        running_acc = running_correct / running_total
        if (i + 1) % log_frequency == 0:
            print(
                f"Training | Epoch: {epoch} | "
                + f"Loss: {running_loss:.4f} | "
                + f"Acc: {running_acc:.2%}"
            )
            running_loss, running_correct, running_total = 0, 0, 0

 82%|████████▏ | 102/125 [00:06<00:01, 15.64it/s]

Training | Epoch: 0 | Loss: 80.3849 | Acc: 48.98%


 82%|████████▏ | 102/125 [00:06<00:01, 15.65it/s]

Training | Epoch: 1 | Loss: 87.8206 | Acc: 49.93%


 82%|████████▏ | 102/125 [00:06<00:01, 15.53it/s]

...

 82%|████████▏ | 102/125 [00:06<00:01, 15.69it/s]

Training | Epoch: 8 | Loss: 86.9382 | Acc: 49.76%


 82%|████████▏ | 102/125 [00:06<00:01, 15.56it/s]

Training | Epoch: 9 | Loss: 86.8762 | Acc: 50.71%


100%|██████████| 125/125 [00:08<00:00, 15.60it/s]


In [17]:

from bioplnn.models import SpatiallyEmbeddedRNN, SpatiallyEmbeddedAreaConfig

area_configs_feedback_model = [
    SpatiallyEmbeddedAreaConfig(
                num_neuron_types = 2,
                num_neuron_subtypes = np.array([16, 16]),
                neuron_type_class = np.array(['excitatory', 'inhibitory']),
                inter_neuron_type_connectivity = np.array([[1, 1, 0], [1, 0, 0], [1, 1, 1], [1, 1, 0]]),
                feedback_channels = 16,
                in_size = [28, 28],
                in_channels =  1,
                out_channels = 32,
    ),
    SpatiallyEmbeddedAreaConfig(
                num_neuron_types = 2,
                num_neuron_subtypes = np.array([32, 32]),
                neuron_type_class = np.array(['excitatory', 'inhibitory']),
                inter_neuron_type_connectivity = np.array([[1, 1, 0], [1, 1, 1], [1, 1, 0]]),
                in_size = [14, 14],
                in_channels =  32,
                out_channels = 32,
    ),
    SpatiallyEmbeddedAreaConfig(
                num_neuron_types = 2,
                num_neuron_subtypes = np.array([32, 32]),
                neuron_type_class = np.array(['excitatory', 'inhibitory']),
                inter_neuron_type_connectivity = np.array([[1, 1, 0], [1, 1, 1], [1, 1, 0]]),
                in_size = [14, 14],
                in_channels =  32,
                out_channels = 32,
    )
]

model_wFeedback = SpatiallyEmbeddedRNN(
                            num_areas = 3,
                            area_configs = area_configs_feedback_model,
                            batch_first = False,
                            inter_area_feedback_connectivity = np.array([[0, 0, 0],[1, 0, 0], [1, 0, 0]])
                    )

In [18]:
model_wFeedback

SpatiallyEmbeddedRNN(
  (areas): ModuleList(
    (0): SpatiallyEmbeddedArea(
      (out_nonlinearity): Identity()
      (neuron_type_nonlinearity): ModuleList(
        (0-1): 2 x Sigmoid()
      )
      (convs): ModuleDict(
        (0->0): Sequential(
          (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): Identity()
        )
        (0->1): Sequential(
          (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): Identity()
        )
        (1->0): Sequential(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): Identity()
        )
        (2->0): Sequential(
          (0): Conv2dRectify(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): Identity()
        )
        (2->1): Sequential(
          (0): Conv2dRectify(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): Identity()
        )
        (3->0): Sequential(
          (